# 03 · Instruction tuning

Instruction tuning = supervised fine-tuning (SFT) on `(instruction → good response)` pairs so a model behaves like an assistant.
Llama-3.2-3B-**Instruct** already went through this; we do a *second* round on our own data to specialise it.

> Needs only `transformers` (tokenizer download). Runs on CPU.

In [ ]:
!pip install -q transformers

## 1. The Llama 3.x chat template
We use the ungated mirror `unsloth/Llama-3.2-3B-Instruct` (the official `meta-llama/Llama-3.2-3B-Instruct` needs accepting Meta's license on Hugging Face).

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B-Instruct")

messages = [
    {"role": "system", "content": "You are a concise, helpful assistant."},
    {"role": "user", "content": "Explain LoRA in one sentence."},
    {"role": "assistant", "content": "LoRA freezes the base model and trains tiny low-rank matrices added to its layers."},
]
text = tok.apply_chat_template(messages, tokenize=False)
print(text)

Key special tokens:
- `<|begin_of_text|>` – start
- `<|start_header_id|>role<|end_header_id|>` – role marker
- `<|eot_id|>` – end of turn (the model must learn to emit this to **stop**)

> **Always train with the same template you will use at inference.** A mismatch is the #1 cause of "fine-tuned model produces garbage".

## 2. Train only on the answer (loss masking)
The loss should be computed on the *assistant* tokens, not the user prompt. Let's see the boundaries.

In [ ]:
ids = tok.apply_chat_template(messages, tokenize=True)
marker = "<|start_header_id|>assistant<|end_header_id|>\n\n"
full = tok.decode(ids)
start = full.index(marker) + len(marker)
print("PROMPT part (labels = -100, ignored):\n", full[:start])
print("RESPONSE part (labels = real tokens, trained):\n", full[start:])

In notebook 05 this is done with Unsloth's `train_on_responses_only`, which sets labels to `-100` for everything before `<|start_header_id|>assistant<|end_header_id|>`.

## 3. Data format we will produce
One JSON object per line (`messages` schema, same as OpenAI/HF):
```json
{"messages":[{"role":"user","content":"..."},{"role":"assistant","content":"..."}]}
```

Next → `04_Data_preparation.ipynb`